This project is based upon a system that takes multiple policy and procedure documents of an organization, reads and understands them and then takes all the PCI DSS requirements as inputs and searches across all documents using AI. At last finds the most relevant matching policy text and tells us exactly where to find the exact policy in the organization's policy and procedure document. 

Process: Documents → Read → Chunk → Embeddings → Search → Compliance → Output

In [154]:
import PyPDF2 #read PDFs
import re
import numpy as np #math (similarity calculation)
from sentence_transformers import SentenceTransformer #AI model for embeddings

In [174]:
model = SentenceTransformer('all-MiniLM-L6-v2') 
#Loads pre-trained AI model, Converts text → numbers (embeddings),Understands meaning, not just keywords

In [175]:
#Reads multiple pdf documents
def read_pdf(file_name):
    data = []
    
    with open(file_name, "rb") as file:
        reader = PyPDF2.PdfReader(file) #reader becomes an object that represents your entire PDF file
        
        #enumerate() is a Python built-in function.It converts a list into: (index, value)
        #reader.pages is a list-like object.It contains all pages in the PDF.Page here is a page object. 
        #Page object is a Python object that represents one single page of a PDF,along with all its content and properties.
        
        #Example : reader.pages = ["page1", "page2", "page3"]
        #for i, page in enumerate(reader.pages):
        #enumerate(reader.pages) becomes: (0, page1), (1, page2), (2, page3)
        
        
        for i, page in enumerate(reader.pages): 
            text = page.extract_text() #Reads the content inside that page.Converts it into text
            
            data.append({
                "doc": file_name,
                "page": i + 1,
                "text": text
            })
    
    return data

In [176]:
#Load all the documents
all_data = []

files = ["01_Information_Security_Policy.pdf", "02_Acceptable_Use_Policy.pdf", "03_Access_Control_Policy.pdf","04_Change_Control_Policy.pdf","05_Configuration_Management_Policy.pdf","06_Data_Handling_and_Encryption_Policy.pdf","07_Hardening_Procedures.pdf","08_Password_Policy.pdf","09_Incident_Response_Policy.pdf","10_Secure_Development_Policy.pdf" ]  # add all your files

for file in files:
    pdf_data = read_pdf(file)
    all_data.extend(pdf_data)

Suppose chunk 1 is:

“SAD not stored after authorization”

and after passing through the model, it becomes an embedding like:

chunk1 = [0.12, 0.44, 0.76]

This means this sentence is now represented as a point in a 3D space (in real life it’s 384D, but we simplify).

Now suppose you have many chunks:

embeddings =
[
 [0.12, 0.44, 0.76],   → chunk 1
 [0.55, 0.21, 0.33],   → chunk 2
 [0.90, 0.10, 0.40]    → chunk 3
]

So this becomes a matrix of shape:

[N × D] = [3 × 3]

Now your query is something like:

“Is SAD stored after authorization?”

After embedding:

query = [0.10, 0.40, 0.70]

Shape:

[1 × D] = [1 × 3]

Now you transpose query:

query.T =
[
 [0.10],
 [0.40],
 [0.70]
]

Shape becomes:

[D × 1] = [3 × 1]

Now we do:

scores = np.dot(embeddings, query.T)

So mathematically:

[3 × 3] · [3 × 1] = [3 × 1]

Now what actually happens internally:

For chunk 1:
[0.12, 0.44, 0.76] · [0.10, 0.40, 0.70]
= (0.12×0.10) + (0.44×0.40) + (0.76×0.70)
= similarity score 1

For chunk 2:
[0.55, 0.21, 0.33] · query
= similarity score 2
For chunk 3:
[0.90, 0.10, 0.40] · query
= similarity score 3

So final output becomes:

scores =
[
 0.68,   → chunk 1 similarity
 0.42,   → chunk 2 similarity
 0.55    → chunk 3 similarity
]

“How is dot product happening for ALL embeddings without a for loop?”

Because NumPy does vectorized matrix multiplication. It automatically applies the operation to ALL rows at once.No loop needed in Python.

In [177]:
#Chunking

import re
def chunk_data(all_data):
    chunks = []

    for item in all_data:
        doc = item["doc"]
        page = item["page"]
        text = item["text"]

        # First split by lines (preserve policy structure)
        lines = text.split("\n")

        for line in lines:
            line = line.strip()

            if len(line.split()) >= 4:
                chunks.append({
                    "doc": doc,
                    "page": page,
                    "text": line
                })
            else:
                # If line is too small, try sentence split
                #?<= is part of a regular expression (regex) and it is called a:Positive Lookbehind Assertion
                #(?<=pattern) : Only match something if it is preceded by pattern
                #Split at spaces ONLY if the space comes after ., !, ?
                sentences = re.split(r'(?<=[.!?])\s+', line) 
                
                for sentence in sentences:
                    if len(sentence.split()) >= 4:
                        chunks.append({
                            "doc": doc,
                            "page": page,
                            "text": sentence
                        })

    return chunks


In [178]:
#Create Chunks
chunks = chunk_data(all_data)

In [179]:
#Create Embeddings
def create_embeddings(chunks):
    texts = [chunk["text"] for chunk in chunks]
    embeddings = model.encode(texts)
    return embeddings

In [180]:
#Generate Embeddings
embeddings = create_embeddings(chunks)

In [181]:
#Search Function
def search(query, chunks, embeddings):
    query_embedding = model.encode([query]) #Converts requirement → vector
    
    scores = np.dot(embeddings, query_embedding.T).flatten() #Compare query with ALL chunks,Compute similarity score
    
    # np.argsort(scores) = sorted from lowest to highest 
    # [::-1] = reversed → highest first
    # [:5] = Take top 5 results
    
    best_indices = np.argsort(scores)[::-1][:5] 
    
    results = []
    
    for i in best_indices:
        results.append({
            "doc": chunks[i]["doc"],
            "page": chunks[i]["page"],
            "text": chunks[i]["text"],
            "score": float(scores[i])
        })
    
    return results

In [182]:
pci_requirements = [
    "PCI DSS Req 3.3.1:SAD is not stored after authorization, even if encrypted",
    "PCI DSS Req 3.4.1:PAN is masked when displayed (the BIN and last four digits are the maximum number of digits to be displayed), such that only personnel with a legitimate business need can see more than the BIN and last four digits of the PAN.",
    "PCI DSS Req 8.3.1:If passwords/passphrases are used as authentication factors to meet Requirement 8.3.1, they meet the following minimum level of complexity. A minimum length of 12 characters (or IF the system does not support 12 characters, a minimum length of eight characters)."
]

In [183]:
def check_compliance(pci_requirements, chunks, embeddings):
    results = []
    
    for req in pci_requirements:
        search_results = search(req, chunks, embeddings)
        
        strong_matches = [r for r in search_results if r["score"] > 0.6]
        
        if strong_matches:
            status = "Compliant"
            reason = "Relevant policy found with high similarity"
        else:
            status = "Not Compliant"
            reason = "No strong policy found"
        
        results.append({
            "requirement": req,
            "status": status,
            "reason": reason,
            "matches": strong_matches
        })
    
    return results

In [184]:
compliance_results = check_compliance(pci_requirements, chunks, embeddings)

In [185]:
def display_results(results):
    for result in results:
        print("\n" + "="*50)
        print("Requirement:", result["requirement"])
        print("Status:", result["status"])
        print("Reason:", result["reason"])
        
        for match in result["matches"]:
            print("\nDocument:", match["doc"])
            print("Page:", match["page"])
            print("Score:", round(match["score"], 2))
            print("Text:", match["text"])

In [188]:
display_results(compliance_results)


Requirement: PCI DSS Req 3.3.1:SAD is not stored after authorization, even if encrypted
Status: Compliant
Reason: Relevant policy found with high similarity

Document: 01_Information_Security_Policy.pdf
Page: 7
Score: 0.67
Text: • SAD shall never be stored after authorization, even if encrypted

Requirement: PCI DSS Req 3.4.1:PAN is masked when displayed (the BIN and last four digits are the maximum number of digits to be displayed), such that only personnel with a legitimate business need can see more than the BIN and last four digits of the PAN.
Status: Compliant
Reason: Relevant policy found with high similarity

Document: 01_Information_Security_Policy.pdf
Page: 7
Score: 0.6
Text: • Full PAN shall be masked when displayed, showing at most the first six

Requirement: PCI DSS Req 8.3.1:If passwords/passphrases are used as authentication factors to meet Requirement 8.3.1, they meet the following minimum level of complexity. A minimum length of 12 characters (or IF the system does not

IF I WANT THE OUTPUT IN AN EXCEL

import sys
!{sys.executable} -m pip install pandas openpyxl

import pandas as pd

def export_to_excel(results):
    rows = []

    for result in results:
        requirement = result["requirement"]
        status = result["status"]
        reason = result["reason"]

        for match in result["matches"]:
            rows.append({
                "Requirement": requirement,
                "Status": status,
                "Reason": reason,
                "Document": match["doc"],
                "Page": match["page"],
                "Score": round(match["score"], 2),
                "Text": match["text"]
            })

    df = pd.DataFrame(rows)

    # Save to Excel
    df.to_excel("PCI_Compliance_Report.xlsx", index=False)

    print("Excel file created: PCI_Compliance_Report.xlsx")
    export_to_excel(compliance_results)

In [152]:
import pandas as pd

def display_table(results):
    rows = []

    for result in results:
        for match in reszult["matches"]:
            rows.append({
                "Requirement": result["requirement"],
                "Status": result["status"],
                "Document": match["doc"],
                "Page": match["page"],
                "Score": round(match["score"], 2),
                "Text": match["text"]
            })

    df = pd.DataFrame(rows)
    return df

In [153]:
df = display_table(compliance_results)
df

,Requirement,Status,Document,Page,Score,Text
0,PCI DSS Req 3.3.1:SAD is not stored after auth...,Compliant,01_Information_Security_Policy.pdf,7,0.67,• SAD shall never be stored after authorizatio...
1,PCI DSS Req 3.4.1:PAN is masked when displayed...,Compliant,01_Information_Security_Policy.pdf,7,0.60,"• Full PAN shall be masked when displayed, sho..."
2,PCI DSS Req 8.3.1:If passwords/passphrases are...,Compliant,08_Password_Policy.pdf,2,0.65,2. Password Requirements2.1 Standard Account P...
3,PCI DSS Req 8.3.1:If passwords/passphrases are...,Compliant,01_Information_Security_Policy.pdf,8,0.61,PCI DSS Requirement 3
4,PCI DSS Req 8.3.1:If passwords/passphrases are...,Compliant,08_Password_Policy.pdf,1,0.61,SecureOrg Inc.PCI DSS Compliance FrameworkPass...
